## Architektura

1) Rownolegle dziala wiele srodowisk, wszystkie sa resetowane na starcie epizodu.
2) Dla kazdego stanu wyznaczana jest maska bezpiecznych akcji.
3) Model z rozdzieleniem wartosci stanu i przewagi akcji w sieci wylicza wartosci Q dla calej paczki stanow.
4) Agent wybiera akcje strategia epsilon-zachlanna z uwzglednieniem maski.
5) Funkcja kroku srodowiska zwraca przejscie (stan, akcja, nagroda, nowy stan, zakonczony epizod).
6) Przejscia trafiaja do bufora zwrotu wielokrokowego, a nastepnie do pamieci z priorytetami.
7) Aktualizacja modelu polega na probkowaniu paczki, obliczeniu straty i kroku optymalizacji.
8) Siec docelowa jest aktualizowana przez lagodne mieszanie wag.
9) Harmonogram zmniejsza wspolczynnik uczenia, a przy spadku wyniku nastepuje powrot do najlepszego punktu kontrolnego.

## Wejscia i wyjscia

- Wejscie: stan siatki i wektor cech pomocniczych; wyjscie: wartosci Q dla trzech akcji.
- Akcje: prosto, w prawo, w lewo.
- Rozmiar wejscia wynika z czterech kanalow planszy i pieciu cech pomocniczych.
- Agent wybiera akcje z najwyzsza wartoscia Q albo losowo.

## Obserwacja dla sieci konwolucyjnej

- Cztery kanaly siatki: wiek ciala, glowa, jedzenie, sciany.
- Piec cech pomocniczych: cztery kierunki oraz dlugosc weza.
- Tensor ma ksztalt 4 x rozmiar_planszy x rozmiar_planszy, po czym jest splaszczany i laczony z wektorem cech pomocniczych.

## Maskowanie akcji

- Usuwa akcje prowadzace do natychmiastowej kolizji ze sciana lub wlasnym cialem.
- Przy duzej zajetosci planszy dodaje krotkie przewidywanie kilku ruchow do przodu.
- Gdy wszystkie akcje sa zle, maska wraca do [True, True, True].

## Pamiec z priorytetami i zwrot wielokrokowy

- Zwrot wielokrokowy scala kilka kolejnych nagrod w jedna wartosc uczenia.
- Priorytet przejscia: $p_i = (|\delta_i| + \epsilon)^{\alpha}$ ([replay_buffer.py](replay_buffer.py#L114)).
- Zwrot wielokrokowy: $R^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k}$ ([replay_buffer.py](replay_buffer.py#L224)).
- Losowanie jest proporcjonalne do priorytetu, a wagi koryguja stronniczosc probkowania.

## Strata i aktualizacja

- Cel z oddzielna siecia docelowa: $y = r + \gamma^n Q_{sieci\_docelowej}(s', arg\,max Q_{sieci\_uczonej})$ ([cnn_agent.py](cnn_agent.py#L223)).
- Funkcja straty z wagami priorytetow: $L = E[w * (Q - y)^2]$ ([cnn_agent.py](cnn_agent.py#L226)).
- Lagodna aktualizacja wag: $\theta_{docelowa} \leftarrow \tau\,\theta_{uczona} + (1-\tau)\,\theta_{docelowa}$ ([cnn_agent.py](cnn_agent.py#L259)).

## Harmonogram wspolczynnika uczenia i cofanie

- Harmonogram zmniejsza wspolczynnik uczenia, gdy metryka nie poprawia sie przez dluzszy czas.
- Cofanie laduje najlepszy punkt kontrolny i resetuje pamiec doswiadczen.
- Po cofnieciu zwiekszana jest eksploracja, a parametry uczenia wracaja do ustawien bazowych.

## Ksztaltowanie nagrody

- Tryby: prosta nagroda, zlozona nagroda oraz mieszanie zalezne od zajetosci planszy.
- Wspolczynnik zajetosci: $occ = dlugosc\_weza / (rozmiar\_planszy * rozmiar\_planszy)$ ([game_model.py](game_model.py#L396)).
- Waga mieszania: $w = (occ - start) / (end - start)$ ([game_model.py](game_model.py#L409)).
- Nagroda mieszana: $reward = (1-w) * prosta + w * zlozona$ ([game_model.py](game_model.py#L626)).

## Przyspieszenia obliczen

- Obliczenia sieci sa wykonywane na procesorze graficznym, co przyspiesza konwolucje i duze paczki.
- Numba przyspiesza przeszukiwanie wszerz w funkcjach bezpieczenstwa.
- Cython przyspiesza probkowanie i aktualizacje w pamieci z priorytetami.